In [10]:
import pandas as pd
import db_mgmt as mgmt
import os
import fix_imports as fi
import shutil

In [11]:
data_schema = 'dbs/canoe_dataset_schema 5.sql'
db_file = 'dbs/canoe_fuel.sqlite'
os.remove(db_file) if os.path.exists(db_file) else None
mgmt.convert_sql_to_sqlite(data_schema, db_file)


Inserting into CostVariable with columns: region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Successfully deleted 1050 row(s) where cost = 0.


In [67]:
db_core = 'dbs/residential/fuel.sqlite'
db_path = f'dbs/canoe_fuel.sqlite'

os.remove(db_path) if os.path.exists(db_path) else None
shutil.copy2(db_core, db_path)



'dbs/canoe_fuel.sqlite'

In [70]:


data = mgmt.sqlite_to_dfs(db_path)

df = data['CostVariable']

dsl = df.loc[(df['tech'].str.contains('dsl', case=False))&(df['region']=='ON')&(df['period']==2025)]
print(dsl)

    region  period      tech  vintage       cost       units  \
240     ON    2025   F_E_DSL     2025  25.029401  2020 M$/PJ   
265     ON    2025   F_I_DSL     2025  24.814443  2020 M$/PJ   
271     ON    2025   F_T_DSL     2025  29.099577  2020 M$/PJ   
272     ON    2025  F_T_RDSL     2025  34.286608  2020 M$/PJ   
279     ON    2025   F_A_DSL     2025  29.099577  2020 M$/PJ   

                notes data_source  dq_cred  dq_geog  dq_struc  dq_tech  \
240            diesel          F1        2        3         2        1   
265            diesel          F1        2        3         2        1   
271            diesel          F1        2        3         2        1   
272  renewable diesel          F1        2        3         2        1   
279            diesel          F1        2        3         2        1   

     dq_time      data_id  
240        1  FUELHRON002  
265        1  FUELHRON002  
271        1  FUELHRON002  
272        1  FUELHRON002  
279        1  FUELHRON002  


In [64]:
db_path = f'dbs/canoe_fuel.sqlite'
cav = mgmt.sqlite_to_dfs(db_path)['CostVariable']

In [109]:
fuel_techs = pd.read_csv('dbs/tech_fuel.csv')
importing = pd.DataFrame()
for fuel in fuel_techs['fuel'].unique():
    df = cav.loc[cav['tech'].isin(fuel_techs[fuel_techs['fuel'] == fuel]['tech'])]
    # print(df)
    for region in df['region'].unique():
        for period in df['period'].unique():
            for vintage in df['vintage'].unique():
            # print(fuel, region, period)
                sub_df = df.loc[(df['region'] == region) & (df['period'] == period) & (df['vintage'] == vintage)]
                if len(sub_df) > 0: 
                    min_cost = sub_df['cost'].min()
                    sub_df.loc[~sub_df['tech'].str.contains('IMP'), 'cost'] = sub_df.loc[~sub_df['tech'].str.contains('IMP'), 'cost'] - min_cost  
                    importing = pd.concat([importing, sub_df])
                    if not sub_df['tech'].str.contains('IMP').any():
                        print(sub_df)



In [110]:
importing[importing['tech'].str.contains('DSL', case=False)]
fuels_cost_var = importing.loc[importing['cost']!=0].copy()

In [111]:
df = fuels_cost_var
df[df['tech'].str.contains('h2', case=False)]

,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
1506,AB,2025,F_T_H2,2025,44.063289,2020 M$/PJ,hydrogen,F1,2,3,2,1,1,FUELHRAB002
2054,AB,2025,F_IMP_H2,2025,12.349702,2020 M$/PJ,hydrogen,F1,2,3,2,1,1,FUELHRAB002
1517,AB,2030,F_T_H2,2030,42.942856,2020 M$/PJ,hydrogen,F1,2,3,2,1,1,FUELHRAB002
2064,AB,2030,F_IMP_H2,2030,14.010297,2020 M$/PJ,hydrogen,F1,2,3,2,1,1,FUELHRAB002
1528,AB,2035,F_T_H2,2035,42.935861,2020 M$/PJ,hydrogen,F1,2,3,2,1,1,FUELHRAB002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2424,PEI,2035,F_IMP_H2,2035,14.090694,2020 M$/PJ,hydrogen,F1,2,3,2,1,1,FUELHRPEI002
2034,PEI,2040,F_T_H2,2040,42.818343,2020 M$/PJ,hydrogen,F1,2,3,2,1,1,FUELHRPEI002
2434,PEI,2040,F_IMP_H2,2040,14.019694,2020 M$/PJ,hydrogen,F1,2,3,2,1,1,FUELHRPEI002
2045,PEI,2045,F_T_H2,2045,42.923447,2020 M$/PJ,hydrogen,F1,2,3,2,1,1,FUELHRPEI002


In [112]:
importing

,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,2025,F_E_BIO_G,2025,0.000000,2020 M$/PJ,gaseous bioenergy,F1,2,3,2,1,1,FUELHRAB002
750,AB,2025,F_IMP_BIO_G,2025,6.042383,2020 M$/PJ,gaseous bioenergy,F1,2,3,2,1,1,FUELHRAB002
15,AB,2030,F_E_BIO_G,2030,0.000000,2020 M$/PJ,gaseous bioenergy,F1,2,3,2,1,1,FUELHRAB002
765,AB,2030,F_IMP_BIO_G,2030,6.042383,2020 M$/PJ,gaseous bioenergy,F1,2,3,2,1,1,FUELHRAB002
30,AB,2035,F_E_BIO_G,2035,0.000000,2020 M$/PJ,gaseous bioenergy,F1,2,3,2,1,1,FUELHRAB002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1469,PEI,2035,F_IMP_PROP,2035,24.633564,2020 M$/PJ,propane,F1,2,3,2,1,1,FUELHRPEI002
734,PEI,2040,F_A_PROP,2040,0.000000,2020 M$/PJ,propane,F1,2,3,2,1,1,FUELHRPEI002
1484,PEI,2040,F_IMP_PROP,2040,25.667066,2020 M$/PJ,propane,F1,2,3,2,1,1,FUELHRPEI002
749,PEI,2045,F_A_PROP,2045,0.000000,2020 M$/PJ,propane,F1,2,3,2,1,1,FUELHRPEI002


In [ ]:
 32
